# Danh gia bo du lieu Website du lich LSA

In [85]:
import os
import nltk
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from whoosh.index import create_in
from whoosh.fields import *
from whoosh.analysis import StandardAnalyzer
from whoosh import qparser
from whoosh import scoring
import whoosh.index as index

import pytrec_eval
import math


nltk.download('punkt_tab')
nltk.download('stopwords')
stoplist = stopwords.words("english")
stoplist.append('oh')
puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
ps = PorterStemmer()
import shutil

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [86]:
import os
import re
import pandas as pd
import numpy as np
import networkx as nx
from tqdm import tqdm
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

In [87]:
from nltk.tokenize import sent_tokenize

def split_sentences(text):
    return sent_tokenize(text)

In [88]:
from nltk.tokenize import sent_tokenize
from pyvi import ViTokenizer

def tokenize_vi_sentence_level(text: str) -> list[str]:
    sentences = sent_tokenize(text)
    tokens = []

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        sent_tokens = ViTokenizer.tokenize(sent)
        tokens.extend(sent_tokens.split())

    return tokens


In [89]:
import re

VI_TOKEN_REGEX = re.compile(
    r"[a-zàáạảãâầấậẩẫăằắặẳẵ"
    r"èéẹẻẽêềếệểễ"
    r"ìíịỉĩ"
    r"òóọỏõôồốộổỗơờớợởỡ"
    r"ùúụủũưừứựửữ"
    r"ỳýỵỷỹđ0-9_]+$"
)

def is_valid_vi_token(token: str) -> bool:
    return bool(VI_TOKEN_REGEX.fullmatch(token))


In [90]:
def load_stopwords(path):
    with open(path, "r", encoding="utf-8") as f:
        stopwords = set(
            line.strip().lower()
            for line in f
            if line.strip()
        )
    return stopwords

STOPWORDS_PATH = "../stopword/vietnamese-stopwords-dash.txt"
vi_stopwords = load_stopwords(STOPWORDS_PATH)

print(f"🛑 Đã load {len(vi_stopwords)} stopword")

🛑 Đã load 1942 stopword


In [91]:
# === Đọc dữ liệu và tiền xử lý ===
import re
import unicodedata

def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    
    # Xóa URL
    text = re.sub(r"http\S+|www\S+", "", text)

    # text = text.lower()

    # # Loại ký tự không cần thiết (giữ chữ, số, dấu câu cơ bản)
    # text = re.sub(r"[^0-9a-zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễ"
    #               r"ìíịỉĩòóọỏõôồốộổỗơờớợởỡ"
    #               r"ùúụủũưừứựửữỳýỵỷỹđ\s.,!?]", " ", text)

    # Chuẩn hóa dấu câu
    text = re.sub(r"[.,!?]+", " ", text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [92]:
def preprocess_query(
    query: str,
    stopwords: set[str] | None = None
) -> list[str]:
    """
    Input : raw query string
    Output: list[token] đã clean + tokenize + remove stopword
    """

    # 1. Clean
    query = clean_text(query)

    # 2. Tokenize
    tokens = tokenize_vi_sentence_level(query)

    # 3. Normalize + filter
    processed_tokens = []
    for tok in tokens:
        tok = tok.lower()

        if not is_valid_vi_token(tok):
            continue

        if tok.isnumeric():
            continue

        if stopwords and tok in stopwords:
            continue

        processed_tokens.append(tok)

    return " ".join(processed_tokens)


In [93]:
STOPWORDS_PATH = "../stopword/vietnamese-stopwords-dash.txt"
vi_stopwords = load_stopwords(STOPWORDS_PATH)

query = "Những địa điểm du lịch nổi tiếng nhất ở Hà Nội là gì?"

tokens = preprocess_query(query, vi_stopwords)

print(tokens)

địa_điểm du_lịch nổi_tiếng hà_nội


In [94]:
import re

def preprocess(tokens):
    """
    tokens: list[str] đã được lọc term
    return: string dùng cho indexing
    """
    tokens = [t.lower() for t in tokens if len(t) > 1]
    return " ".join(tokens)


In [95]:
from whoosh.fields import Schema, TEXT, ID
from whoosh.analysis import StandardAnalyzer

def create_schema():
    return Schema(
        docid=ID(stored=True, unique=True),
        title=TEXT(stored=True, analyzer=StandardAnalyzer()),
        content=TEXT(stored=True, analyzer=StandardAnalyzer())
    )


In [96]:
import shutil
import os

def build_index(index_dir, meta_csv, json_path):
    from whoosh.index import create_in
    from whoosh.fields import Schema

    # 🔥 Xóa index cũ nếu tồn tại
    if os.path.exists(index_dir):
        shutil.rmtree(index_dir)

    os.mkdir(index_dir)

    schema = create_schema()
    ix = create_in(index_dir, schema)
    writer = ix.writer()

    import json
    import pandas as pd

    df = pd.read_csv(meta_csv)

    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    for _, row in df.iterrows():
        docid = str(row["id"])
        title = row["title"]
        doc_file = row["document"]

        tokens = doc_terms.get(doc_file, [])
        if not tokens:
            continue

        content = preprocess(tokens)

        writer.add_document(
            docid=docid,
            title=title,
            content=content
        )

    writer.commit()
    return ix


In [97]:
def readQuery(query_csv):
    df = pd.read_csv(query_csv)
    queries = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        queries[qid] = preprocess_query(row["query"])

    return queries


In [98]:
import ast

def readGroundTruth(query_csv, meta_csv):
    meta = pd.read_csv(meta_csv)
    url2docid = dict(zip(meta["url"], meta["id"]))

    df = pd.read_csv(query_csv)
    qrels = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        qrels[qid] = {}

        urls = ast.literal_eval(row["urls"])
        for u in urls:
            if u in url2docid:
                qrels[qid][str(url2docid[u])] = 1

    return qrels


In [99]:
from gensim import corpora, models

def build_lsa(ix, num_topics=200):
    texts = []
    docids = []

    with ix.searcher() as searcher:
        for doc in searcher.all_stored_fields():
            tokens = doc["content"].split()
            texts.append(tokens)
            docids.append(doc["docid"])

    dictionary = corpora.Dictionary(texts)
    corpus = [dictionary.doc2bow(text) for text in texts]

    tfidf = models.TfidfModel(corpus)
    corpus_tfidf = tfidf[corpus]

    lsa = models.LsiModel(
        corpus_tfidf,
        id2word=dictionary,
        num_topics=num_topics
    )

    corpus_lsa = lsa[corpus_tfidf]

    return dictionary, tfidf, lsa, corpus_lsa, docids


In [100]:
from gensim.similarities import MatrixSimilarity

def lsa_search(query, dictionary, tfidf, lsa, corpus_lsa, docids, top_k=100):
    vec = dictionary.doc2bow(query.split())
    vec_lsa = lsa[tfidf[vec]]

    index = MatrixSimilarity(corpus_lsa)
    sims = index[vec_lsa]

    ranked = sorted(enumerate(sims), key=lambda x: -x[1])[:top_k]

    results = {}
    for rank, (idx, score) in enumerate(ranked):
        results[str(docids[idx])] = float(score)

    return results


In [101]:
def run_lsa_all_queries(queries, lsa_components):
    dictionary, tfidf, lsa, corpus_lsa, docids = lsa_components
    run = {}

    for qid, query in queries.items():
        run[qid] = lsa_search(
            query,
            dictionary,
            tfidf,
            lsa,
            corpus_lsa,
            docids
        )

    return run


In [102]:
def evaluate_set_retrieval_at_k(GroundTruth, RunResults, cutoffs=[5,10,20]):
    """
    GroundTruth: dict {qid: {docid: relevance}}
    RunResults : dict {qid: {docid: score}}
    """

    per_query = {}
    avg_metrics = {k: {"P":0, "R":0, "F1":0} for k in cutoffs}
    n = len(GroundTruth)

    print("========== Per-query results ==========")

    for qid in GroundTruth:
        relevant = set(GroundTruth[qid].keys())

        ranked_docs = sorted(
            RunResults.get(qid, {}).items(),
            key=lambda x: x[1],
            reverse=True
        )

        per_query[qid] = {}

        for k in cutoffs:
            retrieved_k = set(docid for docid, _ in ranked_docs[:k])

            tp = len(relevant & retrieved_k)
            fp = len(retrieved_k) - tp
            fn = len(relevant) - tp

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

            per_query[qid][f"P@{k}"] = precision
            per_query[qid][f"R@{k}"] = recall
            per_query[qid][f"F1@{k}"] = f1

            avg_metrics[k]["P"] += precision
            avg_metrics[k]["R"] += recall
            avg_metrics[k]["F1"] += f1

        print(f"Query {qid}")
        for k in cutoffs:
            print(f"  @ {k}")
            print(f"    Precision : {per_query[qid][f'P@{k}']:.4f}")
            print(f"    Recall    : {per_query[qid][f'R@{k}']:.4f}")
            print(f"    F1        : {per_query[qid][f'F1@{k}']:.4f}")
        print("-" * 30)

    print("\n========== Average over all queries ==========")
    for k in cutoffs:
        print(f"@{k}")
        print(f"  Precision : {avg_metrics[k]['P']/n:.4f}")
        print(f"  Recall    : {avg_metrics[k]['R']/n:.4f}")
        print(f"  F1        : {avg_metrics[k]['F1']/n:.4f}")

    return per_query, avg_metrics


## Bo cac tu it xuat hien

In [103]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. LSA
lsa_components = build_lsa(ix, num_topics=200)
print("✅ Đã xây dựng xong LSA")
# 4. Run
run = run_lsa_all_queries(queries, lsa_components)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã xây dựng xong LSA
✅ Đã chạy xong tất cả truy vấn


In [104]:
queries

{'1': 'nhà_thờ ở kon_tum',
 '2': 'vũng_tàu có các địa_điểm nào đẹp',
 '3': 'những địa_điểm du_lịch nổi_tiếng nhất ở hà_nội là gì',
 '4': 'nên đi đâu khi du_lịch đà_nẵng',
 '5': 'du_lịch hội_an có những trải nghiệm đặc_sắc nào',
 '6': 'thời_điểm lý_tưởng để du_lịch sa_pa là khi nào',
 '7': 'các điểm tham_quan không nên bỏ lỡ khi đến huế',
 '8': 'du_lịch ninh_bình nên đi tràng_an hay tam_cốc',
 '9': 'địa_điểm du_lịch sinh_thái nổi_bật ở miền tây_nam_bộ',
 '10': 'những nơi check in đẹp nhất ở đà_lạt dành cho giới trẻ',
 '11': 'du_lịch hạ_long có những tour và hoạt_động nào hấp_dẫn',
 '12': 'các địa_điểm du_lịch tâm_linh nổi_tiếng ở việt_nam',
 '13': 'nên đi du_lịch côn_đảo vào mùa nào trong năm',
 '14': 'các địa_điểm du_lịch gần tp hcm phù_hợp đi cuối tuần',
 '15': 'du_lịch mộc_châu có gì hấp_dẫn vào mùa hoa',
 '16': 'những vườn quốc_gia đẹp và nổi_tiếng nhất việt_nam',
 '17': 'du_lịch quy_nhơn có những bãi biển hoang_sơ nào',
 '18': 'du_lịch miền huế nên đi đâu',
 '19': 'đồng_nai có nhữn

In [105]:
qrels

{'1': {'494': 1, '495': 1},
 '2': {'9': 1,
  '123': 1,
  '241': 1,
  '242': 1,
  '424': 1,
  '425': 1,
  '426': 1,
  '427': 1},
 '3': {'49': 1,
  '350': 1,
  '351': 1,
  '356': 1,
  '357': 1,
  '358': 1,
  '359': 1,
  '470': 1,
  '471': 1},
 '4': {'37': 1,
  '279': 1,
  '185': 1,
  '280': 1,
  '281': 1,
  '285': 1,
  '286': 1,
  '287': 1,
  '288': 1,
  '289': 1,
  '290': 1,
  '484': 1,
  '485': 1,
  '486': 1,
  '487': 1},
 '5': {'51': 1, '129': 1, '130': 1, '131': 1, '304': 1, '305': 1},
 '6': {'100': 1,
  '188': 1,
  '189': 1,
  '381': 1,
  '382': 1,
  '383': 1,
  '384': 1,
  '462': 1,
  '463': 1,
  '464': 1,
  '465': 1},
 '7': {'114': 1,
  '128': 1,
  '248': 1,
  '249': 1,
  '352': 1,
  '353': 1,
  '354': 1,
  '355': 1,
  '385': 1,
  '386': 1,
  '436': 1,
  '437': 1},
 '8': {'134': 1,
  '135': 1,
  '112': 1,
  '78': 1,
  '246': 1,
  '247': 1,
  '348': 1,
  '349': 1},
 '9': {'399': 1,
  '400': 1,
  '406': 1,
  '407': 1,
  '408': 1,
  '428': 1,
  '429': 1,
  '430': 1,
  '431': 1,
  '43

In [106]:
run

{'1': {'494': 0.923019289970398,
  '362': 0.6718937158584595,
  '363': 0.6366140246391296,
  '57': 0.5684168338775635,
  '314': 0.37976160645484924,
  '67': 0.3544313311576843,
  '495': 0.34482258558273315,
  '232': 0.20661567151546478,
  '172': 0.1365811824798584,
  '333': 0.134171262383461,
  '78': 0.13236956298351288,
  '205': 0.1292019933462143,
  '201': 0.12799936532974243,
  '405': 0.11988507211208344,
  '207': 0.11775216460227966,
  '297': 0.11171301454305649,
  '372': 0.0988442450761795,
  '368': 0.09854798763990402,
  '390': 0.09168875217437744,
  '216': 0.09005285799503326,
  '410': 0.08875228464603424,
  '349': 0.08657164126634598,
  '328': 0.08399148285388947,
  '49': 0.08224540948867798,
  '331': 0.08218298852443695,
  '75': 0.07854245603084564,
  '5': 0.07756301760673523,
  '400': 0.07563918828964233,
  '392': 0.07319121807813644,
  '177': 0.07184183597564697,
  '167': 0.07122799009084702,
  '165': 0.06935560703277588,
  '221': 0.06395798921585083,
  '358': 0.061730228364

In [107]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.2000
    Recall    : 0.5000
    F1        : 0.2857
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.2500
    Recall    : 0.6250
    F1        : 0.3571
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3333
    F1        : 0.4286
  @ 10
    Precision : 0.5000
    Recall    : 0.5556
    F1        : 0.5263
  @ 20
    Precision : 0.3500
    Recall    : 0.7778
    F1        : 0.4828
------------------------------
Query 4
  @ 5
    Precision : 1.0000
    Recall    : 0.3333
    F1        : 0.5000
  @ 10
    Precision : 0.9000
    Recall    : 0.6000
    F1        : 0.7200
  @

In [108]:
import gc
import time

def safe_remove_index(index_dir):
    try:
        gc.collect()
        time.sleep(1)
        shutil.rmtree(index_dir)
    except PermissionError as e:
        print("⚠️ Không xóa được index, hãy restart kernel:", e)


In [109]:
if os.path.exists(INDEX_DIR):
    safe_remove_index(INDEX_DIR)

## Chi bo stopword

In [110]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. LSA
lsa_components = build_lsa(ix, num_topics=200)
print("✅ Đã xây dựng xong LSA")
# 4. Run
run = run_lsa_all_queries(queries, lsa_components)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã xây dựng xong LSA
✅ Đã chạy xong tất cả truy vấn


In [111]:
run

{'1': {'494': 0.9233564138412476,
  '362': 0.6753451824188232,
  '363': 0.6323920488357544,
  '57': 0.568259060382843,
  '314': 0.38934966921806335,
  '67': 0.3461945652961731,
  '495': 0.34370940923690796,
  '232': 0.19525662064552307,
  '78': 0.1411581188440323,
  '333': 0.1344764530658722,
  '205': 0.13243885338306427,
  '201': 0.1306934654712677,
  '172': 0.12700746953487396,
  '405': 0.12348465621471405,
  '207': 0.12317142635583878,
  '297': 0.10828034579753876,
  '368': 0.09887970983982086,
  '372': 0.09286147356033325,
  '390': 0.09155836701393127,
  '216': 0.09114978462457657,
  '410': 0.09064516425132751,
  '349': 0.08796032518148422,
  '328': 0.08758240193128586,
  '331': 0.0846262127161026,
  '49': 0.07953955233097076,
  '5': 0.07860986888408661,
  '400': 0.07587245106697083,
  '392': 0.07060012966394424,
  '221': 0.0697077065706253,
  '165': 0.06771785020828247,
  '167': 0.0637921541929245,
  '177': 0.06329841911792755,
  '75': 0.06327535957098007,
  '358': 0.0601571723818

In [112]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.2000
    Recall    : 0.5000
    F1        : 0.2857
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.2500
    Recall    : 0.6250
    F1        : 0.3571
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3333
    F1        : 0.4286
  @ 10
    Precision : 0.5000
    Recall    : 0.5556
    F1        : 0.5263
  @ 20
    Precision : 0.3500
    Recall    : 0.7778
    F1        : 0.4828
------------------------------
Query 4
  @ 5
    Precision : 0.8000
    Recall    : 0.2667
    F1        : 0.4000
  @ 10
    Precision : 0.9000
    Recall    : 0.6000
    F1        : 0.7200
  @

## Cac term la 1 tu

In [113]:
def preprocess_query_word(
    query: str,
    stopwords: set[str] | None = None
) -> list[str]:
    """
    Input : raw query string
    Output: list[token] đã clean + tokenize + remove stopword
    """

    # 1. Clean
    query = clean_text(query)

    # 2. Tokenize
    tokens = tokenize_vi_sentence_level(query)

    # 3. Normalize + filter
    processed_tokens = []
    for tok in tokens:
        tok = tok.lower()

        if not is_valid_vi_token(tok):
            continue

        if tok.isnumeric():
            continue

        if stopwords and tok in stopwords:
            continue

        processed_tokens.append(tok)

    return " ".join(processed_tokens)


In [114]:
import re

def preprocess_word(tokens):
    """
    tokens: list[str] đã được lọc term
    return: string dùng cho indexing
    """
    processed = []

    for t in tokens:
        t = t.lower()
        if len(t) <= 1:
            continue

        if "_" in t:
            processed.extend(t.split("_"))
        else:
            processed.append(t)

    return " ".join(processed)


In [115]:
import shutil
import os

def build_index_word(index_dir, meta_csv, json_path):
    from whoosh.index import create_in
    from whoosh.fields import Schema

    # 🔥 Xóa index cũ nếu tồn tại
    if os.path.exists(index_dir):
        shutil.rmtree(index_dir)

    os.mkdir(index_dir)

    schema = create_schema()
    ix = create_in(index_dir, schema)
    writer = ix.writer()

    import json
    import pandas as pd

    df = pd.read_csv(meta_csv)

    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    for _, row in df.iterrows():
        docid = str(row["id"])
        title = row["title"]
        doc_file = row["document"]

        tokens = doc_terms.get(doc_file, [])
        if not tokens:
            continue

        content = preprocess_word(tokens)

        writer.add_document(
            docid=docid,
            title=title,
            content=content
        )

    writer.commit()
    return ix


In [116]:
def readQuery_word(query_csv):
    df = pd.read_csv(query_csv)
    queries = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        queries[qid] = preprocess_query_word(row["query"])

    return queries


In [117]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index_word(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery_word(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. LSA
lsa_components = build_lsa(ix, num_topics=200)
print("✅ Đã xây dựng xong LSA")
# 4. Run
run = run_lsa_all_queries(queries, lsa_components)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã xây dựng xong LSA
✅ Đã chạy xong tất cả truy vấn


In [118]:
run

{'1': {'94': 0.38228583335876465,
  '93': 0.34471529722213745,
  '187': 0.2990422248840332,
  '90': 0.2556007504463196,
  '131': 0.25217902660369873,
  '89': 0.25065183639526367,
  '92': 0.23510073125362396,
  '63': 0.23225238919258118,
  '51': 0.2181684523820877,
  '75': 0.21423210203647614,
  '17': 0.20159563422203064,
  '58': 0.1887880265712738,
  '114': 0.18404218554496765,
  '327': 0.17656289041042328,
  '50': 0.1698392778635025,
  '15': 0.16649264097213745,
  '350': 0.1455848515033722,
  '401': 0.14094795286655426,
  '31': 0.13839766383171082,
  '100': 0.13610175251960754,
  '95': 0.13154931366443634,
  '186': 0.12807808816432953,
  '386': 0.12525321543216705,
  '368': 0.12045162171125412,
  '10': 0.11960993707180023,
  '128': 0.10795727372169495,
  '167': 0.10692574828863144,
  '409': 0.1063549816608429,
  '369': 0.10346738994121552,
  '67': 0.0985403060913086,
  '237': 0.09521502256393433,
  '326': 0.09066466987133026,
  '224': 0.09044204652309418,
  '133': 0.08683523535728455,

In [119]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 20
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
------------------------------
Query 2
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 20
    Precision : 0.0500
    Recall    : 0.1250
    F1        : 0.0714
------------------------------
Query 3
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 20
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
------------------------------
Query 4
  @ 5
    Precision : 0.0000
    Recall    : 0.0000
    F1        : 0.0000
  @ 10
    Precision : 0.2000
    Recall    : 0.1333
    F1        : 0.1600
  @

## Them PageRank va Quan trong

In [120]:
def normalize(scores):
    min_s, max_s = min(scores), max(scores)
    return [(s - min_s) / (max_s - min_s + 1e-9) for s in scores]

In [121]:
ALPHA = 0.8

def rank_with_pagerank(run, pagerank_dict):
    """
    run: dict[qid][doc_id] = lsa_score
    pagerank_dict: dict[doc_id] = pagerank
    """
    final_run = {}

    for qid, doc_scores in run.items():
        # lấy danh sách doc_id và score
        doc_ids = list(doc_scores.keys())
        lsa_scores = list(doc_scores.values())

        # lấy pagerank (ép kiểu nếu cần)
        pr_scores = [
            pagerank_dict.get(int(doc_id), 0.0)
            for doc_id in doc_ids
        ]

        # normalize pagerank theo query
        pr_norm = normalize(pr_scores)

        # kết hợp điểm
        reranked = {}
        for doc_id, lsa, pr in zip(doc_ids, lsa_scores, pr_norm):
            score = ALPHA * lsa + (1 - ALPHA) * pr
            reranked[doc_id] = score

        # sort giảm dần
        reranked = dict(
            sorted(reranked.items(), key=lambda x: x[1], reverse=True)
        )

        final_run[qid] = reranked

    return final_run


In [122]:
import pandas as pd

def load_pagerank(meta_csv):
    df = pd.read_csv(meta_csv)

    # giả sử cột là: doc_id, pagerank
    pagerank = dict(zip(df["id"], df["pagerank"]))

    return pagerank

In [123]:
pagerank

{0: 0.0024141259892052,
 1: 0.0047040581141591,
 2: 0.0,
 3: 0.0031371979990231,
 4: 0.0042234751157577,
 5: 0.0,
 6: 0.0,
 7: 0.0061042427410997,
 8: 0.0,
 9: 0.0119431051878678,
 10: 0.0043401776207795,
 11: 0.0009157402443533,
 12: 0.0,
 13: 0.0009157402443533,
 14: 0.0,
 15: 0.0,
 16: 0.0061042427410997,
 17: 0.0047040581141591,
 18: 0.0061042427410997,
 19: 0.0009157402443533,
 20: 0.0009157402443533,
 21: 0.0040039658006888,
 22: 0.0069132042286454,
 23: 0.0061847085512597,
 24: 0.0061042427410997,
 25: 0.0,
 26: 0.004299544888981,
 27: 0.0024141259892052,
 28: 0.0016941218194395,
 29: 0.0040039658006888,
 30: 0.0061042427410997,
 31: 0.0022384710246594,
 32: 0.0009157402443533,
 33: 0.0,
 34: 0.0061042427410997,
 35: 0.0045144551805684,
 36: 0.0061042427410997,
 37: 0.0089046119949809,
 38: 0.0011103356381248,
 39: 0.0061042427410997,
 40: 0.0040039658006888,
 41: 0.0,
 42: 0.0015480182077885,
 43: 0.0,
 44: 0.0061042427410997,
 45: 0.0,
 46: 0.0061042427410997,
 47: 0.006104242

In [124]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# Load PageRank scores
pagerank_dict = load_pagerank(META_CSV)

# 1. Index
ix = build_index(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. LSA
lsa_components = build_lsa(ix, num_topics=200)
print("✅ Đã xây dựng xong LSA")
# 4. Run
run = run_lsa_all_queries(queries, lsa_components)
print("✅ Đã chạy xong tất cả truy vấn")
final_run = rank_with_pagerank(run, pagerank_dict)
print("✅ Đã xếp hạng lại với PageRank")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã xây dựng xong LSA
✅ Đã chạy xong tất cả truy vấn
✅ Đã xếp hạng lại với PageRank


In [125]:
final_run

{'1': {'494': 0.7397202491760254,
  '57': 0.5318559233441679,
  '362': 0.5282247543334961,
  '363': 0.5013154029846192,
  '67': 0.35873895328191197,
  '314': 0.29527988433837893,
  '495': 0.27851402759552,
  '232': 0.26735067723335415,
  '78': 0.23400805756101234,
  '109': 0.22878566197606912,
  '49': 0.18700912193034802,
  '95': 0.1854645901646383,
  '297': 0.1664214510086907,
  '129': 0.1606107155546392,
  '177': 0.15706959431513265,
  '205': 0.15605170272270813,
  '114': 0.15601733788499458,
  '172': 0.14917719202560645,
  '90': 0.14891821309575662,
  '207': 0.14736970804611815,
  '37': 0.14732330008992775,
  '124': 0.13707685476127002,
  '221': 0.13594574968961154,
  '123': 0.13420619648311094,
  '154': 0.1340453502575528,
  '113': 0.12453363924990507,
  '165': 0.12191368237727407,
  '224': 0.12020532827523622,
  '16': 0.11875767748502168,
  '235': 0.117593340403068,
  '201': 0.11394607753659343,
  '47': 0.11261132698205384,
  '333': 0.11064218282699585,
  '76': 0.11049852113870057

In [126]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    final_run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.2000
    Recall    : 0.5000
    F1        : 0.2857
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.2500
    Recall    : 0.6250
    F1        : 0.3571
------------------------------
Query 3
  @ 5
    Precision : 0.2000
    Recall    : 0.1111
    F1        : 0.1429
  @ 10
    Precision : 0.3000
    Recall    : 0.3333
    F1        : 0.3158
  @ 20
    Precision : 0.2500
    Recall    : 0.5556
    F1        : 0.3448
------------------------------
Query 4
  @ 5
    Precision : 1.0000
    Recall    : 0.3333
    F1        : 0.5000
  @ 10
    Precision : 0.7000
    Recall    : 0.4667
    F1        : 0.5600
  @

In [127]:
final_run

{'1': {'494': 0.7397202491760254,
  '57': 0.5318559233441679,
  '362': 0.5282247543334961,
  '363': 0.5013154029846192,
  '67': 0.35873895328191197,
  '314': 0.29527988433837893,
  '495': 0.27851402759552,
  '232': 0.26735067723335415,
  '78': 0.23400805756101234,
  '109': 0.22878566197606912,
  '49': 0.18700912193034802,
  '95': 0.1854645901646383,
  '297': 0.1664214510086907,
  '129': 0.1606107155546392,
  '177': 0.15706959431513265,
  '205': 0.15605170272270813,
  '114': 0.15601733788499458,
  '172': 0.14917719202560645,
  '90': 0.14891821309575662,
  '207': 0.14736970804611815,
  '37': 0.14732330008992775,
  '124': 0.13707685476127002,
  '221': 0.13594574968961154,
  '123': 0.13420619648311094,
  '154': 0.1340453502575528,
  '113': 0.12453363924990507,
  '165': 0.12191368237727407,
  '224': 0.12020532827523622,
  '16': 0.11875767748502168,
  '235': 0.117593340403068,
  '201': 0.11394607753659343,
  '47': 0.11261132698205384,
  '333': 0.11064218282699585,
  '76': 0.11049852113870057